Setup inicial en Python (Colab / Jupyter)

Descargar el dataset con kagglehub

In [ ]:
!pip install kagglehub

import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.io as pio
pio.renderers.default = "colab"


plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)

# Descargar siempre la versión más reciente del dataset
path = kagglehub.dataset_download("rockyt07/formula-1-championships-1950-2025")

print("Path to dataset files:", path)
print("Archivos dentro de la carpeta:")
print(os.listdir(path))

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




Using Colab cache for faster access to the 'formula-1-championships-1950-2025' dataset.
Path to dataset files: /kaggle/input/formula-1-championships-1950-2025
Archivos dentro de la carpeta:
['races.csv', 'drivers.csv', 'constructors.csv', 'driver_standings.csv', 'constructor_standings.csv', 'results.csv', 'circuits.csv', 'qualifying.csv']


Reemplazar las rutas de lectura

In [ ]:
def read_csv_smart(path):
    for enc in ["utf-8", "latin1", "ISO-8859-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Leído OK con encoding: {enc} -> {path}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer el archivo con los encodings probados: {path}")

base = path

circuits = read_csv_smart(f"{base}/circuits.csv")
constructor_standings = read_csv_smart(f"{base}/constructor_standings.csv")
constructors = read_csv_smart(f"{base}/constructors.csv")
driver_standings = read_csv_smart(f"{base}/driver_standings.csv")
drivers = read_csv_smart(f"{base}/drivers.csv")
qualifying = read_csv_smart(f"{base}/qualifying.csv")
races = read_csv_smart(f"{base}/races.csv")
results = read_csv_smart(f"{base}/results.csv")

Leído OK con encoding: latin1 -> /kaggle/input/formula-1-championships-1950-2025/circuits.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/constructor_standings.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/constructors.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/driver_standings.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/drivers.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/qualifying.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/races.csv
Leído OK con encoding: utf-8 -> /kaggle/input/formula-1-championships-1950-2025/results.csv


In [ ]:
# Resumen de formas
dfs = {
    "circuits": circuits,
    "constructors": constructors,
    "drivers": drivers,
    "races": races,
    "results": results,
    "driver_standings": driver_standings,
    "constructor_standings": constructor_standings,
    "qualifying": qualifying,
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

print("\nColumnas clave:")
for name, df in dfs.items():
    print(f"\n{name.upper()} COLUMNS:")
    print(df.columns.tolist())

circuits: (76, 7)
constructors: (168, 4)
drivers: (616, 5)
races: (1149, 7)
results: (7600, 9)
driver_standings: (3131, 6)
constructor_standings: (1061, 6)
qualifying: (3017, 7)

Columnas clave:

CIRCUITS COLUMNS:
['circuit_id', 'name', 'lat', 'long', 'locality', 'country', 'Wikipedia_url ']

CONSTRUCTORS COLUMNS:
['constructor_id', 'name', 'nationality', 'Wikipedia_url']

DRIVERS COLUMNS:
['driver_id', 'givenName', 'familyName', 'nationality', 'dob']

RACES COLUMNS:
['race_id', 'season', 'round', 'race_name', 'date', 'time', 'circuit_id']

RESULTS COLUMNS:
['race_id', 'driver_id', 'constructor_id', 'grid', 'position', 'position_order', 'points', 'laps', 'status']

DRIVER_STANDINGS COLUMNS:
['season', 'round', 'driver_id', 'position', 'points', 'wins']

CONSTRUCTOR_STANDINGS COLUMNS:
['season', 'round', 'constructor_id', 'position', 'points', 'wins']

QUALIFYING COLUMNS:
['race_id', 'driver_id', 'constructor_id', 'position', 'q1', 'q2', 'q3']


Numero de carreras por temporada

In [ ]:
import matplotlib.pyplot as plt

# Número de carreras por temporada
races_per_season = (
    races.groupby("season")["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "num_races"})
)

print("Rango de temporadas:")
print("Mínima temporada:", races_per_season["season"].min())
print("Máxima temporada:", races_per_season["season"].max())

print("\nÚltimas 10 temporadas (para revisar 2016–2025 aprox):")
print(races_per_season.sort_values("season").tail(10))

Rango de temporadas:
Mínima temporada: 1950
Máxima temporada: 2025

Últimas 10 temporadas (para revisar 2016–2025 aprox):
    season  num_races
66    2016         21
67    2017         20
68    2018         21
69    2019         21
70    2020         17
71    2021         22
72    2022         22
73    2023         22
74    2024         24
75    2025         24


Graficamos para obtener mejor visualizacion

In [ ]:
!pip install plotly -q

import plotly.express as px

In [ ]:
fig = px.line(
    races_per_season,
    x="season",
    y="num_races",
    markers=True,  # para ver los puntos
    title="Número de carreras por temporada (F1)",
    labels={
        "season": "Temporada",
        "num_races": "Número de carreras"
    },
)

# Opcional: mejorar el hover (tooltip)
fig.update_traces(
    hovertemplate="Temporada: %{x}<br>Carreras: %{y}<extra></extra>"
)

fig.show()

# KPI del calendario por temporada

In [ ]:
races_per_season = (
    races.groupby("season")["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "num_races"})
    .sort_values("season")
)

# KPIs globales
total_seasons = races_per_season["season"].nunique()
total_races = races["race_id"].nunique()
avg_races = races_per_season["num_races"].mean()
min_races = races_per_season["num_races"].min()
max_races = races_per_season["num_races"].max()

season_min_races = races_per_season.loc[races_per_season["num_races"].idxmin(), "season"]
season_max_races = races_per_season.loc[races_per_season["num_races"].idxmax(), "season"]

print(f"Total de temporadas: {total_seasons}")
print(f"Total de carreras disputadas: {total_races}")
print(f"Carreras promedio por temporada: {avg_races:.2f}")
print(f"Mínimo de carreras en una temporada: {min_races} (season {season_min_races})")
print(f"Máximo de carreras en una temporada: {max_races} (season {season_max_races})")

print("\nÚltimas 10 temporadas (para revisar el cierre del dataset):")
display(races_per_season.tail(10))

Total de temporadas: 76
Total de carreras disputadas: 1149
Carreras promedio por temporada: 15.12
Mínimo de carreras en una temporada: 7 (season 1950)
Máximo de carreras en una temporada: 24 (season 2024)

Últimas 10 temporadas (para revisar el cierre del dataset):


,season,num_races
66,2016,21
67,2017,20
68,2018,21
69,2019,21
70,2020,17
71,2021,22
72,2022,22
73,2023,22
74,2024,24
75,2025,24


# Top 10 pilotos por victorias históricas (con gráfico interactivo)


In [ ]:
import plotly.express as px

# Ganadores por carrera
winners = results[results["position_order"] == 1].copy()

# Contar victorias por piloto
wins_by_driver = (
    winners.groupby("driver_id")["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "wins"})
)

# Crear nombre completo del piloto
drivers["full_name"] = drivers["givenName"] + " " + drivers["familyName"]

wins_by_driver = wins_by_driver.merge(
    drivers[["driver_id", "full_name", "nationality"]],
    on="driver_id",
    how="left"
)

# Top 10
top10_drivers = wins_by_driver.sort_values("wins", ascending=False).head(10)
display(top10_drivers)

,driver_id,wins,full_name,nationality
48,michael_schumacher,29,Michael Schumacher,German
28,hamilton,22,Lewis Hamilton,British
61,prost,17,Alain Prost,French
12,clark,17,Jim Clark,British
46,max_verstappen,15,Max Verstappen,Dutch
74,senna,15,Ayrton Senna,Brazilian
75,stewart,14,Jackie Stewart,British
82,vettel,14,Sebastian Vettel,German
19,fangio,13,Juan Fangio,Argentine
1,alonso,11,Fernando Alonso,Spanish


# Top Ten

In [ ]:
fig = px.bar(
    top10_drivers.sort_values("wins"),  # ordenado para grafo horizontal
    x="wins",
    y="full_name",
    orientation="h",
    title="Top 10 pilotos por victorias en F1",
    labels={
        "wins": "Número de victorias",
        "full_name": "Piloto"
    },
    hover_data={"nationality": True}
)

fig.update_traces(
    hovertemplate="Piloto: %{y}<br>Victorias: %{x}<br>Nacionalidad: %{customdata[0]}<extra></extra>"
)

fig.show()

# Top constructores por victorias

In [ ]:
# 1) Aseguramos el dataframe de ganadores (por si no quedó en memoria)
winners = results[results["position_order"] == 1].copy()

# 2) Contar victorias por constructor
wins_by_constructor = (
    winners.groupby("constructor_id")["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "wins"})
)

# 3) Unir con tabla de constructores para obtener el nombre
wins_by_constructor = wins_by_constructor.merge(
    constructors[["constructor_id", "name", "nationality"]],
    on="constructor_id",
    how="left"
)

# 4) Top 10 constructores
top10_constructors = wins_by_constructor.sort_values("wins", ascending=False).head(10)
top10_constructors

,constructor_id,wins,name,nationality
13,ferrari,84,Ferrari,Italian
24,mclaren,59,McLaren,British
26,mercedes,35,Mercedes,German
28,red_bull,28,Red Bull,Austrian
34,williams,26,Williams,British
30,team_lotus,17,Team Lotus,British
18,lotus-climax,15,Lotus-Climax,British
29,renault,12,Renault,French
7,brm,8,BRM,British
31,tyrrell,8,Tyrrell,British


In [ ]:
fig = px.bar(
    top10_constructors.sort_values("wins"),
    x="wins",
    y="name",
    orientation="h",
    title="Top 10 constructores por victorias en F1",
    labels={
        "wins": "Número de victorias",
        "name": "Constructor"
    },
    hover_data={"nationality": True}
)

fig.update_traces(
    hovertemplate="Constructor: %{y}<br>Victorias: %{x}<br>Nacionalidad: %{customdata[0]}<extra></extra>"
)

fig.show()

# Campeones por temporada

In [ ]:
""" driver_standings ya trae season, round, driver_id, position, points, wins.
Para cada season tomamos el round máximo (última carrera).
En ese round nos quedamos con position == 1 → campeón.
Luego lo unimos con drivers para tener el nombre completo."""
# 1) Identificar el último round de cada temporada en driver_standings
last_round_per_season = (
    driver_standings
    .groupby("season")["round"]
    .max()
    .reset_index()
    .rename(columns={"round": "last_round"})
)

# 2) Unir para quedarnos solo con el último round de cada season
ds_last = driver_standings.merge(
    last_round_per_season,
    on="season",
    how="inner"
)

ds_last = ds_last[ds_last["round"] == ds_last["last_round"]]

# 3) Filtrar campeones (posición 1)
champions = ds_last[ds_last["position"] == 1].copy()

# 4) Añadir nombre del piloto
drivers["full_name"] = drivers["givenName"] + " " + drivers["familyName"]

champions = champions.merge(
    drivers[["driver_id", "full_name", "nationality"]],
    on="driver_id",
    how="left"
)

# 5) Ordenar por temporada y seleccionar columnas clave
champions_per_season = (
    champions[["season", "full_name", "nationality", "points", "wins"]]
    .sort_values("season")
)

champions_per_season.tail(15)

,season,full_name,nationality,points,wins
61,2011,Sebastian Vettel,German,392.0,11
62,2012,Sebastian Vettel,German,281.0,5
63,2013,Sebastian Vettel,German,397.0,13
64,2014,Lewis Hamilton,British,384.0,11
65,2015,Lewis Hamilton,British,381.0,10
66,2016,Nico Rosberg,German,385.0,9
67,2017,Lewis Hamilton,British,363.0,9
68,2018,Lewis Hamilton,British,408.0,11
69,2019,Lewis Hamilton,British,413.0,11
70,2020,Lewis Hamilton,British,347.0,11


In [ ]:
# nº de títulos por piloto

titles_by_driver = (
    champions_per_season
    .groupby(["full_name", "nationality"])["season"]
    .nunique()
    .reset_index()
    .rename(columns={"season": "titles"})
    .sort_values("titles", ascending=False)
)

titles_by_driver.head(10)

,full_name,nationality,titles
25,Michael Schumacher,German,7
22,Lewis Hamilton,British,7
18,Juan Fangio,Argentine,5
34,Sebastian Vettel,German,4
24,Max Verstappen,Dutch,4
0,Alain Prost,French,4
28,Nelson Piquet,Brazilian,3
31,Niki Lauda,Austrian,3
3,Ayrton Senna,Brazilian,3
9,Jack Brabham,Australian,3


# Ranking de circuitos por número de carreras

In [ ]:
# Merge carreras + info de circuitos
races_circuits = races.merge(
    circuits,
    on="circuit_id",
    how="left",
    suffixes=("_race", "_circuit")
)

# Número de carreras por circuito
gp_per_circuit = (
    races_circuits
    .groupby(["circuit_id", "name", "country"])["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "num_races"})
    .sort_values("num_races", ascending=False)
)

gp_per_circuit.head(15)

,circuit_id,name,country,num_races
45,monza,Autodromo Nazionale di Monza,Italy,75
42,monaco,Circuit de Monaco,Monaco,71
63,silverstone,Silverstone Circuit,UK,60
65,spa,Circuit de Spa-Francorchamps,Belgium,58
69,villeneuve,Circuit Gilles Villeneuve,Canada,44
28,interlagos,Interlagos Circuit,Brazil,42
49,nurburgring,Nurburgring,Germany,41
25,hungaroring,Hungaroring,Hungary,40
55,red_bull_ring,Red Bull Ring,Austria,39
24,hockenheimring,Hockenheimring,Germany,37


# gráfico interactivo (top 15 circuitos):


In [ ]:
top_n = 15
top_circuits = gp_per_circuit.head(top_n).sort_values("num_races")

fig = px.bar(
    top_circuits,
    x="num_races",
    y="name",
    orientation="h",
    title=f"Top {top_n} circuitos por número de Grandes Premios",
    labels={
        "num_races": "Número de carreras",
        "name": "Circuito"
    },
    hover_data={"country": True}
)

fig.update_traces(
    hovertemplate="Circuito: %{y}<br>Carreras: %{x}<br>País: %{customdata[0]}<extra></extra>"
)

fig.show()

# Pilotos con más victorias en el circuito más usado

In [ ]:
# 1) Aseguramos df de ganadores
winners = results[results["position_order"] == 1].copy()

# 2) Añadir info de la carrera (season, circuit_id, race_name)
winners_races = winners.merge(
    races[["race_id", "season", "circuit_id", "race_name"]],
    on="race_id",
    how="left"
)

# 3) Añadir info de circuito (nombre, país)
winners_races_circuits = winners_races.merge(
    circuits[["circuit_id", "name", "country"]],
    on="circuit_id",
    how="left"
)

# 4) Añadir nombre del piloto
drivers["full_name"] = drivers["givenName"] + " " + drivers["familyName"]

winners_full = winners_races_circuits.merge(
    drivers[["driver_id", "full_name", "nationality"]],
    on="driver_id",
    how="left"
)

winners_full.head()

,race_id,driver_id,constructor_id,grid,position,position_order,points,laps,status,season,circuit_id,race_name,name,country,full_name,nationality
0,1950_1,farina,alfa,1,1,1,9.0,70,Finished,1950,silverstone,British Grand Prix,Silverstone Circuit,UK,Nino Farina,Italian
1,1950_2,fangio,alfa,1,1,1,9.0,100,Finished,1950,monaco,Monaco Grand Prix,Circuit de Monaco,Monaco,Juan Fangio,Argentine
2,1950_3,parsons,kurtis_kraft,5,1,1,9.0,138,Finished,1950,indianapolis,Indianapolis 500,Indianapolis Motor Speedway,USA,Johnnie Parsons,American
3,1950_4,farina,alfa,2,1,1,9.0,42,Finished,1950,bremgarten,Swiss Grand Prix,Circuit Bremgarten,Switzerland,Nino Farina,Italian
4,1950_5,fangio,alfa,2,1,1,8.0,35,Finished,1950,spa,Belgian Grand Prix,Circuit de Spa-Francorchamps,Belgium,Juan Fangio,Argentine


In [ ]:
#Ranking de circuitos por número de carreras (basado en races)

races_circuits = races.merge(
    circuits[["circuit_id", "name", "country"]],
    on="circuit_id",
    how="left"
)

gp_per_circuit = (
    races_circuits
    .groupby(["circuit_id", "name", "country"])["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "num_races"})
    .sort_values("num_races", ascending=False)
)

gp_per_circuit.head(10)

,circuit_id,name,country,num_races
45,monza,Autodromo Nazionale di Monza,Italy,75
42,monaco,Circuit de Monaco,Monaco,71
63,silverstone,Silverstone Circuit,UK,60
65,spa,Circuit de Spa-Francorchamps,Belgium,58
69,villeneuve,Circuit Gilles Villeneuve,Canada,44
28,interlagos,Interlagos Circuit,Brazil,42
49,nurburgring,Nurburgring,Germany,41
25,hungaroring,Hungaroring,Hungary,40
55,red_bull_ring,Red Bull Ring,Austria,39
24,hockenheimring,Hockenheimring,Germany,37


Grafico Top 15



In [ ]:
top_n = 15
top_circuits = gp_per_circuit.head(top_n).sort_values("num_races")

fig = px.bar(
    top_circuits,
    x="num_races",
    y="name",
    orientation="h",
    title=f"Top {top_n} circuitos por número de Grandes Premios",
    labels={
        "num_races": "Número de carreras",
        "name": "Circuito"
    },
    hover_data={"country": True}
)

fig.update_traces(
    hovertemplate="Circuito: %{y}<br>Carreras: %{x}<br>País: %{customdata[0]}<extra></extra>"
)

fig.show()

In [ ]:
# Aseguramos tipos consistentes por si acaso
winners_full["circuit_id"] = winners_full["circuit_id"].astype(str)

# Circuitos con más carreras CON resultados (es decir, con ganador registrado)
wins_by_circuit = (
    winners_full
    .groupby(["circuit_id", "name", "country"])["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "num_races_with_results"})
    .sort_values("num_races_with_results", ascending=False)
)

wins_by_circuit.head(15)

,circuit_id,name,country,num_races_with_results
26,monaco,Circuit de Monaco,Monaco,35
1,albert_park,Albert Park Grand Prix Circuit,Australia,28
14,imola,Autodromo Internazionale Enzo e Dino Ferrari,Italy,27
6,catalunya,Circuit de Barcelona-Catalunya,Spain,24
16,interlagos,Interlagos Circuit,Brazil,21
11,galvez,Autódromo Oscar y Juan Gálvez,Argentina,20
41,spa,Circuit de Spa-Francorchamps,Belgium,20
2,bahrain,Bahrain International Circuit,Bahrain,20
22,kyalami,Kyalami,South Africa,18
37,sepang,Sepang International Circuit,Malaysia,15


In [ ]:
main_circuit = wins_by_circuit.iloc[0]

main_circuit_id = main_circuit["circuit_id"]
main_circuit_name = main_circuit["name"]
main_circuit_country = main_circuit["country"]

print(f"Circuito con más carreras (según results): {main_circuit_name} ({main_circuit_country})")
print(f"Número de carreras con resultados: {main_circuit['num_races_with_results']}")

Circuito con más carreras (según results): Circuit de Monaco (Monaco)
Número de carreras con resultados: 35


In [ ]:
#Filtramos los ganadores en ese circuito:
winners_main_circuit = winners_full[winners_full["circuit_id"] == main_circuit_id].copy()
print("Filas en winners_main_circuit:", winners_main_circuit.shape[0])
winners_main_circuit.head()

Filas en winners_main_circuit: 35


,race_id,driver_id,constructor_id,grid,position,position_order,points,laps,status,season,circuit_id,race_name,name,country,full_name,nationality
1,1950_2,fangio,alfa,1,1,1,9.0,100,Finished,1950,monaco,Monaco Grand Prix,Circuit de Monaco,Monaco,Juan Fangio,Argentine
25,1955_2,trintignant,ferrari,9,1,1,8.0,100,Finished,1955,monaco,Monaco Grand Prix,Circuit de Monaco,Monaco,Maurice Trintignant,French
30,1956_2,moss,maserati,2,1,1,8.0,100,Finished,1956,monaco,Monaco Grand Prix,Circuit de Monaco,Monaco,Stirling Moss,British
35,1957_2,fangio,maserati,1,1,1,9.0,105,Finished,1957,monaco,Monaco Grand Prix,Circuit de Monaco,Monaco,Juan Fangio,Argentine
41,1958_2,trintignant,cooper,5,1,1,8.0,100,Finished,1958,monaco,Monaco Grand Prix,Circuit de Monaco,Monaco,Maurice Trintignant,French


In [ ]:
#pilotos dominantes
wins_main_circuit_by_driver = (
    winners_main_circuit
    .groupby(["driver_id", "full_name", "nationality"])["race_id"]
    .nunique()
    .reset_index()
    .rename(columns={"race_id": "wins"})
    .sort_values("wins", ascending=False)
)

wins_main_circuit_by_driver.head(10)

,driver_id,full_name,nationality,wins
4,hill,Graham Hill,British,5
16,senna,Ayrton Senna,Brazilian,4
11,moss,Stirling Moss,British,3
12,prost,Alain Prost,French,3
10,michael_schumacher,Michael Schumacher,German,3
3,fangio,Juan Fangio,Argentine,2
17,stewart,Jackie Stewart,British,2
18,trintignant,Maurice Trintignant,French,2
2,depailler,Patrick Depailler,French,1
0,alonso,Fernando Alonso,Spanish,1


In [ ]:
# Top 10 pilotos dominantes en el circuito seleccionado
top10_dominantes = wins_main_circuit_by_driver.head(10).sort_values("wins")

fig = px.bar(
    top10_dominantes,
    x="wins",
    y="full_name",
    orientation="h",
    title="Pilotos con más victorias en el circuito seleccionado",
    labels={
        "wins": "Número de victorias",
        "full_name": "Piloto"
    },
    hover_data={"nationality": True}
)

fig.update_traces(
    hovertemplate="Piloto: %{y}<br>Victorias: %{x}<br>Nacionalidad: %{customdata[0]}<extra></extra>"
)

fig.show()

# Función: top de pilotos por circuito


In [ ]:
def top_drivers_by_circuit(circuit_query, top_k=10):
    """
    Devuelve el top de pilotos con más victorias en un circuito.
    circuit_query: texto a buscar en el nombre del circuito (case-insensitive).
    top_k: cuántos pilotos mostrar.
    """

    # Buscar circuitos cuyo nombre contenga el texto dado
    matches = circuits[circuits["name"].str.contains(circuit_query, case=False, na=False)].copy()

    if matches.empty:
        print(f"No se encontró ningún circuito que contenga: '{circuit_query}'")
        return None

    print("Circuitos encontrados que coinciden con la búsqueda:")
    display(matches[["circuit_id", "name", "country"]])

    # Usamos todos los circuit_id que coinciden (por si hay variantes de nombre)
    circuit_ids = matches["circuit_id"].astype(str).unique()

    # Filtrar winners_full solo a esos circuitos
    subset = winners_full[winners_full["circuit_id"].isin(circuit_ids)].copy()

    if subset.empty:
        print("No hay resultados de carrera asociados a estos circuitos en 'results'.")
        return None

    # Agrupar victorias por piloto
    wins_by_driver = (
        subset
        .groupby(["driver_id", "full_name", "nationality"])["race_id"]
        .nunique()
        .reset_index()
        .rename(columns={"race_id": "wins"})
        .sort_values("wins", ascending=False)
    )

    top = wins_by_driver.head(top_k)

    print(f"\nTop {len(top)} pilotos por victorias en circuitos que coinciden con: '{circuit_query}'")
    display(top)

    # Piloto con más victorias (el #1)
    best = top.iloc[0]
    print(
        f"\nPiloto con más victorias: {best['full_name']} "
        f"({best['nationality']}) con {best['wins']} victorias."
    )

    return top

In [ ]:
# Mónaco
top_drivers_by_circuit("Monaco")


Circuitos encontrados que coinciden con la búsqueda:


,circuit_id,name,country
1,monaco,Circuit de Monaco,Monaco



Top 10 pilotos por victorias en circuitos que coinciden con: 'Monaco'


,driver_id,full_name,nationality,wins
4,hill,Graham Hill,British,5
16,senna,Ayrton Senna,Brazilian,4
11,moss,Stirling Moss,British,3
12,prost,Alain Prost,French,3
10,michael_schumacher,Michael Schumacher,German,3
3,fangio,Juan Fangio,Argentine,2
17,stewart,Jackie Stewart,British,2
18,trintignant,Maurice Trintignant,French,2
2,depailler,Patrick Depailler,French,1
0,alonso,Fernando Alonso,Spanish,1



Piloto con más victorias: Graham Hill (British) con 5 victorias.


,driver_id,full_name,nationality,wins
4,hill,Graham Hill,British,5
16,senna,Ayrton Senna,Brazilian,4
11,moss,Stirling Moss,British,3
12,prost,Alain Prost,French,3
10,michael_schumacher,Michael Schumacher,German,3
3,fangio,Juan Fangio,Argentine,2
17,stewart,Jackie Stewart,British,2
18,trintignant,Maurice Trintignant,French,2
2,depailler,Patrick Depailler,French,1
0,alonso,Fernando Alonso,Spanish,1


In [ ]:
# Silverstone
top_drivers_by_circuit("Silverstone")

Circuitos encontrados que coinciden con la búsqueda:


,circuit_id,name,country
0,silverstone,Silverstone Circuit,UK



Top 8 pilotos por victorias en circuitos que coinciden con: 'Silverstone'


,driver_id,full_name,nationality,wins
1,clark,Jim Clark,British,3
0,ascari,Alberto Ascari,Italian,1
2,coulthard,David Coulthard,British,1
3,farina,Nino Farina,Italian,1
4,gonzalez,José Froilán González,Argentine,1
5,hamilton,Lewis Hamilton,British,1
6,max_verstappen,Max Verstappen,Dutch,1
7,stewart,Jackie Stewart,British,1



Piloto con más victorias: Jim Clark (British) con 3 victorias.


,driver_id,full_name,nationality,wins
1,clark,Jim Clark,British,3
0,ascari,Alberto Ascari,Italian,1
2,coulthard,David Coulthard,British,1
3,farina,Nino Farina,Italian,1
4,gonzalez,José Froilán González,Argentine,1
5,hamilton,Lewis Hamilton,British,1
6,max_verstappen,Max Verstappen,Dutch,1
7,stewart,Jackie Stewart,British,1


In [ ]:
# Si quieres más filas, por ejemplo top 20:
top_drivers_by_circuit("Monaco", top_k=20)

Circuitos encontrados que coinciden con la búsqueda:


,circuit_id,name,country
1,monaco,Circuit de Monaco,Monaco



Top 19 pilotos por victorias en circuitos que coinciden con: 'Monaco'


,driver_id,full_name,nationality,wins
4,hill,Graham Hill,British,5
16,senna,Ayrton Senna,Brazilian,4
11,moss,Stirling Moss,British,3
12,prost,Alain Prost,French,3
10,michael_schumacher,Michael Schumacher,German,3
3,fangio,Juan Fangio,Argentine,2
17,stewart,Jackie Stewart,British,2
18,trintignant,Maurice Trintignant,French,2
2,depailler,Patrick Depailler,French,1
0,alonso,Fernando Alonso,Spanish,1



Piloto con más victorias: Graham Hill (British) con 5 victorias.


,driver_id,full_name,nationality,wins
4,hill,Graham Hill,British,5
16,senna,Ayrton Senna,Brazilian,4
11,moss,Stirling Moss,British,3
12,prost,Alain Prost,French,3
10,michael_schumacher,Michael Schumacher,German,3
3,fangio,Juan Fangio,Argentine,2
17,stewart,Jackie Stewart,British,2
18,trintignant,Maurice Trintignant,French,2
2,depailler,Patrick Depailler,French,1
0,alonso,Fernando Alonso,Spanish,1


from matplotlib import pyplot as plt
_df_0['wins'].plot(kind='hist', bins=20, title='wins')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['wins'].plot(kind='line', figsize=(8, 4), title='wins')
plt.gca().spines[['top', 'right']].set_visible(False)

La función hace todo esto por ti:
Busca el circuito por nombre parcial (no necesitas escribirlo perfecto).
Te muestra qué circuitos encontró (por transparencia).
Filtra las carreras ganadas en esos circuitos.
Agrupa por piloto y cuenta victorias.
Muestra el top de pilotos y te dice explícitamente quién es el que más ganó ahí.

# campeones por decada

In [ ]:
# Último round por temporada
last_round_per_season = (
    driver_standings
    .groupby("season")["round"]
    .max()
    .reset_index()
    .rename(columns={"round": "last_round"})
)

ds_last = driver_standings.merge(
    last_round_per_season,
    on="season",
    how="inner"
)

ds_last = ds_last[ds_last["round"] == ds_last["last_round"]]

# Campeones (posición 1)
champions = ds_last[ds_last["position"] == 1].copy()

# Nombre completo
drivers["full_name"] = drivers["givenName"] + " " + drivers["familyName"]

champions = champions.merge(
    drivers[["driver_id", "full_name", "nationality"]],
    on="driver_id",
    how="left"
)

champions_per_season = (
    champions[["season", "full_name", "nationality", "points", "wins"]]
    .sort_values("season")
)

champions_per_season.head()

,season,full_name,nationality,points,wins
0,1950,Nino Farina,Italian,30.0,3
1,1951,Juan Fangio,Argentine,31.0,3
2,1952,Alberto Ascari,Italian,36.0,6
3,1953,Alberto Ascari,Italian,34.5,5
4,1954,Juan Fangio,Argentine,42.0,6


In [ ]:
# Crear la columna de década
# Década de cada temporada (1950 → 1950s, 1987 → 1980s, etc.)
champions_per_season["decade_start"] = (champions_per_season["season"] // 10) * 10
champions_per_season["decade"] = champions_per_season["decade_start"].astype(str) + "s"

champions_per_season[["season", "decade", "full_name"]].head(15)

,season,decade,full_name
0,1950,1950s,Nino Farina
1,1951,1950s,Juan Fangio
2,1952,1950s,Alberto Ascari
3,1953,1950s,Alberto Ascari
4,1954,1950s,Juan Fangio
5,1955,1950s,Juan Fangio
6,1956,1950s,Juan Fangio
7,1957,1950s,Juan Fangio
8,1958,1950s,Mike Hawthorn
9,1959,1950s,Jack Brabham


In [ ]:
# Títulos totales por década (visión global)
titles_by_decade = (
    champions_per_season
    .groupby("decade")["season"]
    .nunique()
    .reset_index()
    .rename(columns={"season": "num_titles"})
    .sort_values("decade")
)

titles_by_decade

,decade,num_titles
0,1950s,10
1,1960s,10
2,1970s,10
3,1980s,10
4,1990s,10
5,2000s,10
6,2010s,10
7,2020s,6


In [ ]:
# Cómo se reparten los títulos entre pilotos en cada década
# Conteo de títulos por piloto y década

titles_by_driver_decade = (
    champions_per_season
    .groupby(["decade", "full_name", "nationality"])["season"]
    .nunique()
    .reset_index()
    .rename(columns={"season": "titles"})
    .sort_values(["decade", "titles"], ascending=[True, False])
)

titles_by_driver_decade.head(20)

,decade,full_name,nationality,titles
2,1950s,Juan Fangio,Argentine,5
0,1950s,Alberto Ascari,Italian,2
1,1950s,Jack Brabham,Australian,1
3,1950s,Mike Hawthorn,British,1
4,1950s,Nino Farina,Italian,1
6,1960s,Graham Hill,British,2
7,1960s,Jack Brabham,Australian,2
9,1960s,Jim Clark,British,2
5,1960s,Denny Hulme,New Zealander,1
8,1960s,Jackie Stewart,British,1


In [ ]:
# Función para generar iniciales a partir del nombre completo
def get_initials(name: str) -> str:
    parts = name.split()
    return "".join(p[0].upper() for p in parts if p)

titles_by_driver_decade["initials"] = titles_by_driver_decade["full_name"].apply(get_initials)

# Etiqueta para la leyenda: "Nombre Completo (INICIALES)"
titles_by_driver_decade["legend_label"] = (
    titles_by_driver_decade["full_name"] + " (" + titles_by_driver_decade["initials"] + ")"
)

In [ ]:


# Aseguramos columnas auxiliares
def get_initials(name: str) -> str:
    parts = name.split()
    return "".join(p[0].upper() for p in parts if p)

titles_by_driver_decade["initials"] = titles_by_driver_decade["full_name"].apply(get_initials)
titles_by_driver_decade["legend_label"] = (
    titles_by_driver_decade["full_name"] + " (" + titles_by_driver_decade["initials"] + ")"
)

# ⚠️ OJO: usamos custom_data para poder usarlo en el hovertemplate
fig_decade_pilots = px.bar(
    titles_by_driver_decade,
    x="decade",
    y="titles",
    color="legend_label",       # esto define la leyenda
    text="initials",            # texto encima de la barra
    custom_data=["legend_label"],  # aquí pasamos el texto que queremos en el hover
    title="Distribución de títulos mundiales por piloto y década",
    labels={
        "decade": "Década",
        "titles": "Número de títulos",
        "legend_label": "Piloto"
    },
    barmode="stack"
)

fig_decade_pilots.update_traces(
    textposition="inside",
    insidetextanchor="middle"
)

# Aquí usamos customdata[0] en vez de legend_label
fig_decade_pilots.update_traces(
    hovertemplate=(
        "Década: %{x}<br>"
        "Piloto: %{customdata[0]}<br>"
        "Títulos: %{y}<extra></extra>"
    )
)

fig_decade_pilots.show()
